In [1]:
import sqlite3
conn = sqlite3.connect(":memory:")
cur = conn.cursor()



In [2]:
import pandas as pd

**Create Source Table**

In [5]:
conn.execute("""CREATE TABLE athlete_source (
    athlete_id INTEGER,
    name TEXT,
    updated_at TEXT
);
""")


**Insert initial data:**

In [6]:
conn.execute(
    """
    INSERT INTO athlete_source VALUES
  (1, 'Rahul',  '2024-01-01'),
  (2, 'Amit',   '2024-01-01'),
  (3, 'Neeraj', '2024-01-02');
    """
)

In [8]:
pd.read_sql(
    """
    SELECT * FROM athlete_source;
    """,conn
)

,athlete_id,name,updated_at
0,1,Rahul,2024-01-01
1,2,Amit,2024-01-01
2,3,Neeraj,2024-01-02


**Create Dimension Table (Target)**

In [9]:
conn.execute("""CREATE TABLE athlete_dim (
    athlete_id INTEGER PRIMARY KEY,
    name TEXT,
    updated_at TEXT
);
""")


In [10]:
conn.execute(
    """
    INSERT INTO athlete_dim (athlete_id, name, updated_at)
    SELECT athlete_id, name, updated_at
    FROM athlete_source;
    """
)

In [13]:
pd.read_sql(
    """
    SELECT * FROM athlete_dim ORDER BY athlete_id;
    """,conn
)

,athlete_id,name,updated_at
0,1,Rahul,2024-01-01
1,2,Amit,2024-01-01
2,3,Neeraj,2024-01-02


**WATERMARK LOGIC**
Watermark = “where did I stop last time?”

In [14]:
pd.read_sql(
    """
    SELECT MAX(updated_at) FROM athlete_dim;
    """,conn
)

,MAX(updated_at)
0,2024-01-02


**INCREMENTAL INSERT**

Now simulate new data:

In [15]:
conn.execute(
    """
    INSERT INTO athlete_source VALUES
    (4, 'Karan', '2024-01-03');

    """
)

In [17]:
conn.execute(
    """
    INSERT INTO athlete_dim(athlete_id, name, updated_at)
    SELECT s.athlete_id, s.name, s.updated_at
    FROM athlete_source s
    LEFT JOIN athlete_dim d
      ON s.athlete_id = d.athlete_id
    WHERE d.athlete_id IS NULL
    """
)

In [18]:
pd.read_sql(
    """
    SELECT * FROM athlete_dim ORDER BY athlete_id;

    """,conn
)

,athlete_id,name,updated_at
0,1,Rahul,2024-01-01
1,2,Amit,2024-01-01
2,3,Neeraj,2024-01-02
3,4,Karan,2024-01-03


**INCREMENTAL UPDATE (SCD TYPE 1)**

Now simulate a correction:

In [22]:
conn.execute(
    """
    UPDATE athlete_source
    SET name = 'Neeraj Chopra',
    updated_at = '2024-01-05'
    WHERE athlete_id = 3;

    """
)